In [6]:
# ReLU関数
import numpy as np

def relu(x):
    return np.maximum(x, 0)

def deriv_relu(x):
    return (x > 0).astype(x.dty)

In [7]:
# Sigmoid関数
def sigmoid(x):
    return np.exp(np.minimum(x, 0)) / (1 + np.exp(- np.abs(x)))

def deriv_sigmoid(x):
    return sigmoid(x) * (1 - sigmoid(x))

In [8]:
def np_log(x):
    return np.log(np.clip(x, 1e-10, 1e+10))

In [9]:
def train_xor(x, t, eps):
    """
    :param x: np.ndarray, input data, (batch_size, input_dim)
    :param t: np.ndarray, ground-truth labels, (batch_size, output_dim)
    :param eps: float, learning rate
    """
    global W1, b1, W2, b2

    batch_size = x.shape[0]

    # Forward propagation
    # np.matmul()：入力xと重みW1の行列積を計算
    # バイアスb1を足す
    u1 = np.matmul(x, W1) + b1  # (batch_size, hidden_dim)
    # 前の層で求めた値をReLU関数に通して、隠れ層の出力h1を求める
    h1 = relu(u1)

    # 隠れ層h1を入力として、重みW2とバイアスb2を使ったAffine変換を行う
    u2 = np.matmul(h1, W2) + b2  # (batch_size, output_dim)
    # シグモイド関数に通して、出力を確率に変換
    y = sigmoid(u2)


    # Compute loss
    # 2クラス交差エントロピーの計算
    cost = (- t * np_log(y) - (1 - t) * np_log(1 - y)).mean()

    # Backpropagation
    # シグモイド関数の逆伝播
    # 予測値から正解データを引く
    delta_2 = y - t  
    # ReLU関数の逆伝播
    # deriv_relu(u1)：ReLUの微分
    # W2.T：重みW2の転置行列
    delta_1 = deriv_relu(u1) * np.matmul(delta_2, W2.T)  

    # Compute gradients
    # 1層目の重みとバイアスの逆伝播
    dW1 = np.matmul(x.T, delta_1) / batch_size  
    db1 = np.matmul(np.ones(batch_size), delta_1) / batch_size   

    # 2層目の重みとバイアスの逆伝播
    dW2 = np.matmul(h1.T, delta_2) / batch_size  
    db2 = np.matmul(np.ones(batch_size), delta_2) / batch_size   

    # Update parameters
    # パラメータの更新
    W1 -= eps * dW1 
    b1 -= eps * db1 

    W2 -= eps * dW2 
    b2 -= eps * db2 

    return cost

def valid_xor(x, t):
    global W1, b1, W2, b2

    # Forward propagation
    u1 = np.matmul(x, W1) + b1
    h1 = relu(u1)

    u2 = np.matmul(h1, W2) + b2
    y = sigmoid(u2)

    # Compute loss
    cost = (- t * np_log(y) - (1 - t) * np_log(1 - y)).mean() 

    return cost, y

In [10]:
def softmax(x):
    x -= x.max(axis=1, keepdims=True)  # Avoid overflow
    x_exp = np.exp(x)
    return x_exp / np.sum(x_exp, axis=1, keepdims=True)


def deriv_softmax(x):
    return softmax(x) * (1 - softmax(x))

In [11]:
class Dense:
    # in_dim：入力の次元数
    # out_dim：出力の次元数
    # function：活性化関数
    # deriv_function：活性化関数の微分
    def __init__(self, in_dim, out_dim, function, deriv_function):
        # np.random.uniform：一様分布から乱数を生成するメソッド
        self.W = np.random.uniform(low=-0.08, high=0.08,
                                   size=(in_dim, out_dim)).astype("float64")
        # バイアスの初期化、0で埋める
        self.b = np.zeros(out_dim).astype("float64")
        # この層で使う活性化関数とその微分をインスタンス変数として保持
        self.function = function
        self.deriv_function = deriv_function
        
        # x：入力データを保持
        self.x = None
        # u：活性化関数を通す前の計算結果を保持
        self.u = None

        # 逆伝播で計算される重みとバイアスの勾配を保持
        self.dW = None
        self.db = None

        # ([self.W.size, self.b.size])：重みの要素数とバイアスの要素数を並べたリストを作成
        # np.cumsum()：累積和を計算する関数。リストの値を順番に足し合わせる
        self.params_idxs = np.cumsum([self.W.size, self.b.size])

    def __call__(self, x):
        """
        Method that performs forward propagation.
        x: (batch_size, in_dim_{j})
        h: (batch_size, out_dim_{j})
        """
        # 入力データxをインスタンス変数に保持（逆伝播で使うから）
        self.x = x
        # 全結合層の計算部分
        # 入力xと重みWの行列積をとって、バイアスbを足す
        # self.u：活性化関数を通す前の線形結合として保持
        self.u = np.matmul(self.x, self.W) + self.b
        # 活性化関数を適用して出力hを作成
        h = self.function(self.u)
        # hを次の層へ
        return h

    def b_prop(self, delta, W):
        """
        Method that performs backpropagation.
        delta (=delta_{j+1}): (batch_size, out_dim_{j+1})
        W (=W_{j+1}): (out_dim_{j}, out_dim_{j+1})
        self.delta (=delta_{j}): (batch_size, out_dim_{j})
        """
        # deriv_function：活性化関数の微分
        # ➡ 順伝播の時に保持しておいたself.uを使って、活性化関数の微分を求める
        # np.matmul(delta, W.T)：後ろから流れてきた勾配に重みの転置行列を掛ける
        self.delta = self.deriv_function(self.u) * np.matmul(delta, W.T) 
        return self.delta

    def compute_grad(self):
        """
        Method that computes gradients.
        self.x: (batch_size, in_dim_{j})
        self.delta: (batch_size, out_dim_{j})
        self.dW: (in_dim_{j}, out_dim_{j})
        self.db: (out_dim_{j})
        """
        batch_size = self.delta.shape[0]

        # 重みとバイアスの勾配を求める
        self.dW = np.matmul(self.x.T, self.delta) / batch_size 
        self.db = np.matmul(np.ones(batch_size), self.delta) / batch_size 

    def get_params(self):
        # self.W.ravel()：行列であるself.Wをrevel()を使って一次元の長いベクトルに変換
        # self.b：バイアスはもともと一次元なのでそのまま
        # np.concatenate([], axis=0)：潰した重みとバイアスを縦(axis=0)に繋ぎ合わせている
        return np.concatenate([self.W.ravel(), self.b], axis=0)

    def set_params(self, params):
        """
        params: List[np.ndarray, np.ndarray]
            The first element is the weight matrix W: (in_dim, out_dim), and the second element is the bias vector: (out_dim,)
        """
        # params_idxs：1本の長いベクトルを仕切り地点で切り分ける
        _W, _b = np.split(params, self.params_idxs)[:-1]
        # 切り分けた_Wはまだravel()された1次元の状態だから、元の行列の形（id_dim, out_dim）に戻す
        self.W = _W.reshape(self.W.shape)
        # バイアスは元々1次元だからそのまま
        self.b = _b

    def get_grads(self):
        # self.dW.ravel()：重みの勾配を1次元にし、self.dbと連結して、全パラメータの勾配ベクトルを作って返す
        return np.concatenate([self.dW.ravel(), self.db], axis=0)

In [12]:
class Model:
    def __init__(self, hidden_dims, activation_functions, deriv_functions):
        """
        :param hiden_dims: List[int], a list containing the number of units in each layer.
        :params activation_functions: List, a list of activation functions used in each layer.
        :params derive_functions: List, a list of derivatives of the activation functions used in each layer.
        """
        # Store each layer in a list
        # 作成した全レイヤを格納するための箱を用意
        self.layers = []
        # 出力層の直前までループを回してDenceレイヤを順番に作成
        # hidden_dim[i]：入力次元
        # hidden_dim[i+1]：出力次元
        # activation_functions[i]：活性化関数
        # deriv_functions[i]：活性化関数の微分
        for i in range(len(hidden_dims)-2):  # Same structure for all layers except the output layer
            self.layers.append(Dense(hidden_dims[i], hidden_dims[i+1],
                                     activation_functions[i], deriv_functions[i]))
        # 最後に出力層を追加
        self.layers.append(Dense(hidden_dims[-2], hidden_dims[-1],
                                 activation_functions[-1], deriv_functions[-1]))  # Add the output layer

    # forward処理を呼び出すためのショートカットキー
    def __call__(self, x):
        return self.forward(x)

    # 順伝播の計算部分
    def forward(self, x):
        """Method that performs forward propagation"""
        # ポリモーフィズム
        # ➡ 同じ命令を送っても、対象が違えば勝手に適切な動きをしてくれる仕組み
        # forwardメソッドを指定しなくても、self.layersと書くだけでforward処理を実行してくれる
        for layer in self.layers:
            x = layer(x)
        return x

    def backward(self, delta):
        """Method that performs backpropagation and computes gradients"""
        batch_size = delta.shape[0]

        # [::-1]：リストを逆順にする
        for i, layer in enumerate(self.layers[::-1]):
            # 最初の層は出力層だから前の層から伝わってきた誤差はない
            # その代わり、正解と予測の誤差（delta）が渡されるから、最初の層だけは
            # 渡された値をそのまま自分の誤差として使う処理が必要
            # ➡ それをif文でやる
            if i == 0:  # Output layer
                layer.delta = delta  # y - t
                layer.compute_grad()
            # 後ろの層から伝わってきた誤差を自分の重みを使って変換し、
            # 次の層へ流す処理が必要
            else:  # Non-output layers
                delta = layer.b_prop(delta, W)  # Backpropagation
                layer.compute_grad()  # Compute gradients

            W = layer.W

    def update(self, eps=0.01):
        """Method that updates parameters"""
        for layer in self.layers:
            layer.W -= eps * layer.dW
            layer.b -= eps * layer.db

In [13]:
def train_mst(model, x, t, eps=0.01):
    # Forward propagation
    # 順伝播
    # 入力データからモデルが予測
    y = model(x)

    # Compute loss
    # 損失関数
    # 交差エントロピーの式を使って、正解とどれくらいズレてるかを数値化
    cost = (-t * np_log(y)).sum(axis=1).mean()

    # Backpropagation
    # 誤差の算出
    # 正解と予測の差分（delta）を計算
    # 逆伝播の一歩目
    delta = y - t
    # 計算したdeltaをモデルに入れる
    # 上流から下流に向かって勾配が計算されていく
    model.backward(delta)

    # Update parameters
    # 重みの更新
    # 計算された勾配をもとに、重みを少しずつ修正する
    model.update(eps)

    return cost

In [14]:
def valid_mst(model, x, t):
    # Forward propagation
    y = model(x)

    # Compute loss
    cost = (-t * np_log(y)).sum(axis=1).mean()

    return cost, y

In [ ]:
# 全パラメータの現状把握
def get_params(layers):
    params_all = []
    # for文で全レイヤを周り、それぞれの層が持っている重みをparams_allにまとめる
    for layer in layers:
        params = layer.get_params()
        params_all.append(params)

    return params_all

In [ ]:
# 重みの書き換え
def set_params(layers, params_all):
    # zip(model.layers, params_all)：複数の層と複数の重みをペアにして、一気に書き換え
    for layer, params in zip(model.layers, params_all):
        layer.set_params(params)

In [20]:
# 今の実力のスコア確認
# 現在のモデルがどれくらい間違っているのか、交差エントロピーを計算して、
# 今の実力を測る
def compute_cost(x, t):
    # Forward propagation
    y = model(x)

    # Compute loss
    cost = (-t * np_log(y)).sum(axis=1).mean()

    return cost

In [21]:
batch_size = 32

x = x_train_mnist[:batch_size].astype("float64")
t = t_train_mnist[:batch_size].astype("float64")

NameError: name 'x_train_mnist' is not defined

In [ ]:
# 微小な変化量を定義
eps = 1e-5

# モデルの全層からパラメータを全て取得し、リストとして保持
params_all = get_params(model.layers)
# コレカラ計算する各層の勾配を格納するための空リスト
grads_all_num = []

# Compute gradients for each layer
# layerとparamsをペアにして、層の数だけくり返し処理を行う
for layer, params in zip(model.layers, params_all):
    # パラメータと同じ形状の、中身が全てゼロの配列を作成
    shift = np.zeros_like(params)
    # その層の勾配を格納するための箱を作る
    grads_num = np.zeros_like(params)

    # Compute numerical derivatives for each of the M parameters in the layer
    # その層に含まれるパラメータの要素数分だけループを回し、1つずつ勾配を計算する
    for m in range(len(params)):
        # m番目のパラメータだけをeps分変化させる設定にする
        shift[m] = eps  # Shift only the m-th parameter by eps: [0, 0, ..., 0, eps, 0, ..., 0]

        # 元のパラメータにepsを足した状態を作る
        params_right = params + shift
        # モデルのパラメータをparams_rightに書き換える
        layer.set_params(params_right)
        # 損失関数を計算
        cost_right = compute_cost(x, t)  # L(x; ..., theta_m + eps, ...)

        params_left = params - shift
        layer.set_params(params_left)
        cost_left = compute_cost(x, t)  # L(x; ..., theta_m - eps, ...)

        # 勾配を算出
        grads_num[m] = (cost_right - cost_left) / (2 * eps)  # Compute derivative

        # 計算が終わったら、層のパラメータを元の値に戻す
        layer.set_params(params)
        # shift配列のm番目を0に戻し、次のループの準備
        shift[m] = 0

    # その層の勾配の計算結果をリストに追加
    grads_all_num.append(grads_num)